In [1]:
import gradio as gr
import torch
from transformers import pipeline
import base64
import os

# 1. CARGA DEL MODELO (Actualizado para evitar advertencias)
model_id = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
pipe = pipeline(
    "text-generation", 
    model=model_id, 
    model_kwargs={"dtype": torch.bfloat16}, # 'dtype' en lugar de 'torch_dtype'
    device_map="auto"
)

# 2. CONOCIMIENTO DEL MANUAL
HINTON_KNOWLEDGE = """
Eres la IA de la HINTON 1 (FACI-UPCH). Estilo Cyberpunk. 
Hardware: RTX A6000. Reglas: 10GB cuota, 3 meses inactividad = suspensión.
"""

def chat_func(message, history):
    messages = [
        {"role": "system", "content": HINTON_KNOWLEDGE},
        {"role": "user", "content": message},
    ]
    out = pipe(messages, max_new_tokens=512, do_sample=True, temperature=0.6)
    return out[0]['generated_text'][-1]['content']

# 3. TRATAMIENTO DEL FONDO
def get_base64_image(image_name):
    path = os.path.join("/app", image_name)
    if os.path.exists(path):
        with open(path, "rb") as img_file:
            return base64.b64encode(img_file.read()).decode('utf-8')
    return None

base64_img = get_base64_image("fondo.jpg")
img_data = f"data:image/jpeg;base64,{base64_img}" if base64_img else ""

# 4. DISEÑO DE INTERFAZ
cyberpunk_css = f"""
.gradio-container {{
    background: linear-gradient(rgba(0,0,0,0.7), rgba(0,0,0,0.7)), url('{img_data}') !important;
    background-size: cover !important;
    background-position: center !important;
}}
#chatbot {{ background: rgba(15, 15, 35, 0.9) !important; border: 1px solid #00f2ff !important; }}
h1 {{ color: #00f2ff !important; text-shadow: 0 0 10px #00f2ff; text-align: center; }}
"""

# 5. LANZAMIENTO (Ajustado para Gradio 6.0)
with gr.Blocks() as demo:
    gr.HTML("<h1>HINTON 1 // NEURAL INTERFACE</h1>")
    gr.ChatInterface(fn=chat_func, fill_height=True)

# Pasamos el CSS aquí para evitar el UserWarning
demo.launch(server_name="0.0.0.0", server_port=5051, inline=True, css=cyberpunk_css)

Device set to use cuda:0


* Running on local URL:  http://0.0.0.0:5051
* To create a public link, set `share=True` in `launch()`.
